In [1]:
# Load engineered data (rebuild from raw CSV via src.feature_engineering if missing)
from pathlib import Path
import sys

_root = Path("..").resolve()
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from src.feature_engineering import classify_signal, load_engineered_frames

btc_df, ada_df = load_engineered_frames()
print(f" Loaded engineered data: BTC {btc_df.shape}, ADA {ada_df.shape}")

 Loaded engineered data: BTC (2344, 49), ADA (2344, 42)


In [2]:
from src.feature_engineering import get_feature_columns

# XGBoost is scale-invariant, so we wouldn't strictly need scaling. We still
# expose the feature columns the rest of the notebook uses. We deliberately
# avoid any global fit_transform on the dataframe (that would leak future
# statistics into earlier folds when paired with walk-forward validation).
btc_feature_cols = get_feature_columns(btc_df)
ada_feature_cols = get_feature_columns(ada_df)
print(f"Feature columns: BTC={len(btc_feature_cols)}, ADA={len(ada_feature_cols)}")

Feature columns: BTC=43, ADA=36


# XGBoost Pipeline

Same walk-forward protocol as `2a_lr_model_pipeline`. XGBoost is scale-invariant so `StandardScaler` is skipped. Labels encoded {-1, 0, 1} -> {0, 1, 2} for `multi:softprob`. `RandomizedSearchCV` with `TimeSeriesSplit` tunes hyperparameters per fold.

In [3]:
# =============================================================================
# SECTION 1: IMPORTS & DEPENDENCIES
# =============================================================================
import requests
from datetime import datetime, timedelta
import time
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict
import numpy as np
import pandas as pd

# Shared config
horizons = [3, 7, 14, 30]
thresholds = {
    "fixed_0.5%": 0.005,
    "fixed_1%": 0.01,
    "fixed_2%": 0.02,
}

# ML Libraries
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)
from sklearn.model_selection import RandomizedSearchCV
from xgboost import XGBClassifier


# ============================================================================
# LABEL ENCODING/DECODING FOR XGBOOST (multi:softprob requires 0..n-1)
# ============================================================================
LABEL_ENCODE_MAP = {-1: 0, 0: 1, 1: 2}  # Sell, Hold, Buy -> 0, 1, 2
LABEL_DECODE_MAP = {0: -1, 1: 0, 2: 1}  # 0, 1, 2 -> Sell, Hold, Buy


def encode_labels(y):
    return np.array([LABEL_ENCODE_MAP[int(label)] for label in y])


def decode_labels(y_encoded):
    return np.array([LABEL_DECODE_MAP[int(label)] for label in y_encoded])

In [4]:
from sklearn.model_selection import TimeSeriesSplit
from src.feature_engineering import BINARY_COLUMNS


def _to_np(X):
    return X.to_numpy() if hasattr(X, "to_numpy") else np.asarray(X)


def walk_forward_validation(
    X,
    y,
    model_instance,
    param_dist,
    initial_train_size=0.6,
    step=30,
    n_iter=2,
    cv=3,
    scoring='f1_macro',
):
    """Walk-forward validation with RandomizedSearchCV (TimeSeriesSplit inner CV).

    Phase 0: also returns per-fold std of F1 (std_f1_*) so a single mean is never
    reported on its own. A std comparable to the mean signals an unstable signal.
    """
    X_np = _to_np(X)
    y_np = np.asarray(y)
    n = len(X_np)
    train_end = int(initial_train_size * n)
    results = []
    for start in range(train_end, n, step):
        X_train_np, X_test_np = X_np[:start], X_np[start:start + step]
        y_train_np, y_test_np = y_np[:start], y_np[start:start + step]
        if len(X_test_np) == 0:
            break

        y_train_encoded = encode_labels(y_train_np)

        inner_cv = TimeSeriesSplit(n_splits=cv)
        search = RandomizedSearchCV(
            model_instance, param_dist, n_iter=n_iter, cv=inner_cv,
            scoring=scoring, random_state=42,
        )
        search.fit(X_train_np, y_train_encoded)
        model = search.best_estimator_

        y_pred_encoded = model.predict(X_test_np)
        y_pred = decode_labels(y_pred_encoded)
        precision, recall, fscore, _ = precision_recall_fscore_support(
            y_test_np, y_pred, labels=[-1, 0, 1], zero_division=0,
        )
        results.append({
            'precision_sell': precision[0], 'recall_sell': recall[0], 'f1_sell': fscore[0],
            'precision_hold': precision[1], 'recall_hold': recall[1], 'f1_hold': fscore[1],
            'precision_buy':  precision[2], 'recall_buy':  recall[2], 'f1_buy':  fscore[2],
        })
    if not results:
        return {k: 0 for k in [
            'avg_precision_sell', 'avg_recall_sell', 'avg_f1_sell',
            'avg_precision_hold', 'avg_recall_hold', 'avg_f1_hold',
            'avg_precision_buy',  'avg_recall_buy',  'avg_f1_buy',
            'std_f1_sell', 'std_f1_hold', 'std_f1_buy', 'n_folds',
        ]}
    return {
        'avg_precision_sell': float(np.mean([r['precision_sell'] for r in results])),
        'avg_recall_sell':    float(np.mean([r['recall_sell']    for r in results])),
        'avg_f1_sell':        float(np.mean([r['f1_sell']        for r in results])),
        'avg_precision_hold': float(np.mean([r['precision_hold'] for r in results])),
        'avg_recall_hold':    float(np.mean([r['recall_hold']    for r in results])),
        'avg_f1_hold':        float(np.mean([r['f1_hold']        for r in results])),
        'avg_precision_buy':  float(np.mean([r['precision_buy']  for r in results])),
        'avg_recall_buy':     float(np.mean([r['recall_buy']     for r in results])),
        'avg_f1_buy':         float(np.mean([r['f1_buy']         for r in results])),
        'std_f1_sell':        float(np.std([r['f1_sell']         for r in results])),
        'std_f1_hold':        float(np.std([r['f1_hold']         for r in results])),
        'std_f1_buy':         float(np.std([r['f1_buy']          for r in results])),
        'n_folds':            int(len(results)),
    }


def walk_forward_with_predictions(
    X,
    y,
    model_instance,
    param_dist,
    initial_train_size=0.6,
    step=30,
    n_iter=2,
    cv=3,
    scoring='f1_macro',
):
    """Walk-forward variant that returns aggregated predictions for Binary MCC."""
    X_np = _to_np(X)
    y_np = np.asarray(y)
    fold_metrics = defaultdict(list)
    all_y_true, all_y_pred = [], []
    n = len(X_np)
    idx = int(initial_train_size * n)
    fold_count = 0
    while idx < n:
        X_train_np, X_test_np = X_np[:idx], X_np[idx:idx + step]
        y_train_np, y_test_np = y_np[:idx], y_np[idx:idx + step]
        if len(X_test_np) == 0:
            break

        y_train_encoded = encode_labels(y_train_np)

        inner_cv = TimeSeriesSplit(n_splits=cv)
        search = RandomizedSearchCV(
            model_instance, param_dist, n_iter=n_iter, cv=inner_cv,
            scoring=scoring, random_state=42,
        )
        try:
            search.fit(X_train_np, y_train_encoded)
            model = search.best_estimator_
        except Exception as e:
            print(f"Fold {fold_count + 1} failed: {e}")
            idx += step
            continue

        y_pred_encoded = model.predict(X_test_np)
        y_pred_fold = decode_labels(y_pred_encoded)
        all_y_true.extend(y_test_np.tolist())
        all_y_pred.extend(y_pred_fold.tolist())
        prec, rec, f1, _ = precision_recall_fscore_support(
            y_test_np, y_pred_fold, labels=[-1, 0, 1], average=None, zero_division=0,
        )
        for i, label_name in enumerate(["sell", "hold", "buy"]):
            fold_metrics[f'precision_{label_name}'].append(prec[i])
            fold_metrics[f'recall_{label_name}'].append(rec[i])
            fold_metrics[f'f1_{label_name}'].append(f1[i])
        fold_count += 1
        idx += step
    avg_metrics = {key: np.mean(values) for key, values in fold_metrics.items()}
    return avg_metrics, all_y_true, all_y_pred

In [5]:
# ============================================================================
# Simple MCC utilities (standard + weighted)
# ============================================================================
def _mcc_from_counts(tp, tn, fp, fn):
    import math
    denom = (tp+fp)*(tp+fn)*(tn+fp)*(tn+fn)
    if denom == 0:
        return 0.0
    return ((tp*tn) - (fp*fn)) / math.sqrt(denom)


def evaluate_signal_quality(y_true, y_pred, verbose=True, opposite_weight=2.0):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    def eval_side(pos_label):
        # Binary mapping for MCC
        y_tb = (y_true == pos_label).astype(int)
        y_pb = (y_pred == pos_label).astype(int)
        tp = int(((y_tb == 1) & (y_pb == 1)).sum())
        tn = int(((y_tb == 0) & (y_pb == 0)).sum())
        fp = int(((y_tb == 0) & (y_pb == 1)).sum())
        fn = int(((y_tb == 1) & (y_pb == 0)).sum())
        mcc = _mcc_from_counts(tp, tn, fp, fn)

        # Weighted errors: opposite-direction gets higher penalty
        if pos_label == 1:
            opposite_fp = int(((y_true == -1) & (y_pred == 1)).sum())  # predicted Buy when true Sell
            opposite_fn = int(((y_true == 1) & (y_pred == -1)).sum())  # predicted Sell when true Buy
            miss_from_hold = int(((y_true == 1) & (y_pred == 0)).sum())
            false_from_hold = int(((y_true == 0) & (y_pred == 1)).sum())
        else:  # pos_label == -1 (Sell)
            opposite_fp = int(((y_true == 1) & (y_pred == -1)).sum())  # predicted Sell when true Buy
            opposite_fn = int(((y_true == -1) & (y_pred == 1)).sum())  # predicted Buy when true Sell
            miss_from_hold = int(((y_true == -1) & (y_pred == 0)).sum())
            false_from_hold = int(((y_true == 0) & (y_pred == -1)).sum())

        w_fp = opposite_weight*opposite_fp + false_from_hold
        w_fn = opposite_weight*opposite_fn + miss_from_hold
        w_mcc = _mcc_from_counts(tp, tn, w_fp, w_fn)

        details = {
            'tp': tp, 'tn': tn, 'fp': fp, 'fn': fn,
            'opposite_fp': opposite_fp, 'opposite_fn': opposite_fn,
            'miss_from_hold': miss_from_hold, 'false_from_hold': false_from_hold,
            'mcc': round(mcc, 4), 'weighted_mcc': round(w_mcc, 4)
        }
        return details

    buy = eval_side(1)
    sell = eval_side(-1)

    if verbose:
        total = len(y_true)
        print("\n============================================================")
        print("Binary MCC for Buy Signal (Class 1)")
        print("============================================================")
        print(f"MCC: {buy['mcc']}")
        print("Weighted MCC components (Buy):")
        print(f"  [OK] TP: {buy['tp']}, [OK] TN: {buy['tn']}")
        print(f"  [X] Opposite FP (pred Buy | true Sell): {buy['opposite_fp']} (x{opposite_weight})")
        print(f"  [X] Opposite FN (pred Sell | true Buy): {buy['opposite_fn']} (x{opposite_weight})")
        print(f"  [X] Miss from Hold (true Buy | pred Hold): {buy['miss_from_hold']}")
        print(f"  [X] False Buy from Hold (true Hold | pred Buy): {buy['false_from_hold']}")
        print(f"Weighted MCC (Buy): {buy['weighted_mcc']}")

        print("\n============================================================")
        print("Binary MCC for Sell Signal (Class -1)")
        print("============================================================")
        print(f"MCC: {sell['mcc']}")
        print("Weighted MCC components (Sell):")
        print(f"  [OK] TP: {sell['tp']}, [OK] TN: {sell['tn']}")
        print(f"  [X] Opposite FP (pred Sell | true Buy): {sell['opposite_fp']} (x{opposite_weight})")
        print(f"  [X] Opposite FN (pred Buy | true Sell): {sell['opposite_fn']} (x{opposite_weight})")
        print(f"  [X] Miss from Hold (true Sell | pred Hold): {sell['miss_from_hold']}")
        print(f"  [X] False Sell from Hold (true Hold | pred Sell): {sell['false_from_hold']}")
        print(f"Weighted MCC (Sell): {sell['weighted_mcc']}")

    return {
        'buy_mcc': buy['mcc'],
        'buy_weighted_mcc': buy['weighted_mcc'],
        'sell_mcc': sell['mcc'],
        'sell_weighted_mcc': sell['weighted_mcc'],
    }

## Binary MCC Evaluation

**Buy/Sell signals** evaluated as independent binary classifiers. MCC (-1 to +1) handles class imbalance better than accuracy or F1 alone. The **weighted variant** penalises opposite-direction errors (Buy predicted when true is Sell, or vice versa) at 2x relative to missed signals into Hold.

In [6]:
%%time
# ============================================================================
# SECTION 5A: BITCOIN (BTC) - WALK-FORWARD VALIDATION
# ============================================================================
btc_results = []

xgb_model = XGBClassifier(
    objective="multi:softprob",
    eval_metric="mlogloss",
    random_state=42,
    tree_method="hist",
)
xgb_param_dist = {
    "n_estimators": [200, 400],
    "learning_rate": [0.05, 0.1],
    "max_depth": [3, 5, 7],
    "subsample": [0.7, 1.0],
    "colsample_bytree": [0.7, 1.0],
}

for h in horizons:
    for name, t in thresholds.items():
        threshold = t
        btc_df[f"signal_{h}d_{name}"] = btc_df[f"fwd_return_{h}d"].apply(lambda x: classify_signal(x, threshold))
        feature_cols = get_feature_columns(btc_df)
        X = btc_df[feature_cols]
        y = btc_df[f"signal_{h}d_{name}"]
        combined = pd.concat([X, y], axis=1).dropna()
        X_clean = combined[feature_cols]
        y_clean = combined[f"signal_{h}d_{name}"].astype(int)
        if len(X_clean) < 100:
            print(f"BTC {h}d {name}: Only {len(X_clean)} samples, skipping...")
            continue
        avg_metrics = walk_forward_validation(X_clean, y_clean, xgb_model, xgb_param_dist, cv=3)
        btc_results.append({
            "crypto": "BTC",
            "horizon_days": h,
            "threshold_type": name,
            "threshold_value": round(threshold, 4),
            **avg_metrics,
        })

btc_results_df = pd.DataFrame(btc_results)

CPU times: total: 4h 38min 29s
Wall time: 21min 17s


In [7]:
%%time
# ============================================================================
# SECTION 5B: CARDANO (ADA) - WALK-FORWARD VALIDATION
# ============================================================================
ada_results = []

xgb_model = XGBClassifier(
    objective="multi:softprob",
    eval_metric="mlogloss",
    random_state=42,
    tree_method="hist",
)
xgb_param_dist = {
    "n_estimators": [200, 400],
    "learning_rate": [0.05, 0.1],
    "max_depth": [3, 5, 7],
    "subsample": [0.7, 1.0],
    "colsample_bytree": [0.7, 1.0],
}

for h in horizons:
    for name, t in thresholds.items():
        threshold = t
        ada_df[f"signal_{h}d_{name}"] = ada_df[f"fwd_return_{h}d"].apply(lambda x: classify_signal(x, threshold))
        feature_cols = get_feature_columns(ada_df)
        X = ada_df[feature_cols]
        y = ada_df[f"signal_{h}d_{name}"]
        combined = pd.concat([X, y], axis=1).dropna()
        X_clean = combined[feature_cols]
        y_clean = combined[f"signal_{h}d_{name}"].astype(int)
        if len(X_clean) < 100:
            print(f"ADA {h}d {name}: Only {len(X_clean)} samples, skipping...")
            continue
        avg_metrics = walk_forward_validation(X_clean, y_clean, xgb_model, xgb_param_dist, cv=3)
        ada_results.append({
            "crypto": "ADA",
            "horizon_days": h,
            "threshold_type": name,
            "threshold_value": round(threshold, 4),
            **avg_metrics,
        })

ada_results_df = pd.DataFrame(ada_results)

CPU times: total: 4h 13min 52s
Wall time: 17min 57s


In [8]:
# ============================================================================
# SECTION 6: COMBINED RESULTS & SUMMARY
# ============================================================================
# Merge BTC and ADA results for comprehensive comparison
# Sorted by cryptocurrency, time horizon, and threshold value for readability
# ============================================================================

# Combine results from both cryptocurrencies
all_results_df = pd.concat([btc_results_df, ada_results_df], ignore_index=True)

# Sort for clear presentation
# - crypto: Group Bitcoin and Cardano separately
# - horizon_days: Compare results across time horizons
# - threshold_value: Compare results across threshold configurations
all_results_df.sort_values(["crypto", "horizon_days", "threshold_value"], inplace=True)

print("\n" + "=" * 80)
print("XGBOOST MODEL RESULTS")
print("=" * 80)
print(f"Total Configurations Evaluated: {len(all_results_df)}")
print(f"  - Cryptocurrencies: 2 (BTC, ADA)")
print(f"  - Horizons: 4 (3, 7, 14, 30 days)")
print(f"  - Thresholds: 3 (fixed_0.5%, fixed_1%, fixed_2%)")
print("=" * 80 + "\n")

all_results_df



XGBOOST MODEL RESULTS
Total Configurations Evaluated: 24
  - Cryptocurrencies: 2 (BTC, ADA)
  - Horizons: 4 (3, 7, 14, 30 days)
  - Thresholds: 3 (fixed_0.5%, fixed_1%, fixed_2%)



,crypto,horizon_days,threshold_type,threshold_value,avg_precision_sell,avg_recall_sell,avg_f1_sell,avg_precision_hold,avg_recall_hold,avg_f1_hold,avg_precision_buy,avg_recall_buy,avg_f1_buy,std_f1_sell,std_f1_hold,std_f1_buy,n_folds
12,ADA,3,fixed_0.5%,0.005,0.537072,0.610204,0.519831,0.000000,0.000000,0.000000,0.394785,0.414087,0.352112,0.168353,0.000000,0.217428,29
13,ADA,3,fixed_1%,0.010,0.529626,0.593684,0.503191,0.034169,0.039409,0.029830,0.367513,0.473277,0.370033,0.188888,0.100685,0.233702,29
14,ADA,3,fixed_2%,0.020,0.457751,0.484863,0.416287,0.193471,0.178027,0.168312,0.298543,0.391837,0.300605,0.163107,0.199965,0.186053,29
15,ADA,7,fixed_0.5%,0.005,0.635029,0.722273,0.620039,0.000000,0.000000,0.000000,0.453525,0.467950,0.386575,0.205257,0.000000,0.255275,29
16,ADA,7,fixed_1%,0.010,0.600676,0.716298,0.590543,0.000000,0.000000,0.000000,0.457225,0.496916,0.406003,0.223220,0.000000,0.258000,29
17,ADA,7,fixed_2%,0.020,0.564259,0.735085,0.579232,0.064368,0.043295,0.050000,0.385597,0.430158,0.341733,0.204213,0.113293,0.250030,29
18,ADA,14,fixed_0.5%,0.005,0.610814,0.666219,0.569410,0.000000,0.000000,0.000000,0.402233,0.427800,0.341720,0.262572,0.000000,0.261085,29
19,ADA,14,fixed_1%,0.010,0.583799,0.692734,0.573747,0.017241,0.013793,0.015326,0.447064,0.430288,0.349326,0.280726,0.081096,0.243584,29
20,ADA,14,fixed_2%,0.020,0.604884,0.668299,0.552800,0.101724,0.075657,0.077194,0.429396,0.465066,0.365632,0.262417,0.155029,0.286058,29
21,ADA,30,fixed_0.5%,0.005,0.607294,0.614070,0.506790,0.000000,0.000000,0.000000,0.418114,0.336828,0.280414,0.305298,0.000000,0.290622,28


In [9]:
# ============================================================================
# PHASE 0: 24-config averages + spread (never a single number)
# ============================================================================
print("=" * 80)
print("PHASE 0 SUMMARY - XGBoost, 24 configurations")
print("=" * 80)
for side in ["buy", "sell"]:
    m = all_results_df[f"avg_f1_{side}"].mean()
    s = all_results_df[f"avg_f1_{side}"].std()
    fold = all_results_df[f"std_f1_{side}"].median()
    print(f"{side.title():4} F1: {m:.4f} +/- {s:.4f} across configs | "
          f"median within-config fold-std {fold:.4f}")

best = all_results_df.loc[all_results_df["avg_f1_sell"].idxmax()]
print(f"\nBest Sell-F1 config: {best['crypto']} {int(best['horizon_days'])}d "
      f"{best['threshold_type']} -> Sell F1 {best['avg_f1_sell']:.4f} "
      f"(within-config fold-std {best['std_f1_sell']:.4f})")
print("Reminder: best-of-24 is partly luck; treat as a multiple-testing candidate, "
      "not a confirmed edge (see VALIDATION_PLAN Phase 2.3).")


PHASE 0 SUMMARY - XGBoost, 24 configurations
Buy  F1: 0.3734 +/- 0.0710 across configs | median within-config fold-std 0.2566
Sell F1: 0.4353 +/- 0.1257 across configs | median within-config fold-std 0.2470

Best Sell-F1 config: ADA 7d fixed_0.5% -> Sell F1 0.6200 (within-config fold-std 0.2053)
Reminder: best-of-24 is partly luck; treat as a multiple-testing candidate, not a confirmed edge (see VALIDATION_PLAN Phase 2.3).


## Section 7: Feature Importance Analysis

Identify which on-chain metrics are most predictive of buy/sell/hold signals. Computed for the primary configuration only (30d horizon, 1% threshold) to keep runtime reasonable.

In [10]:
# ============================================================================
# FEATURE IMPORTANCE - XGBoost Gain from Walk-Forward Validation
# ============================================================================
# OPTIMIZATION: Only compute feature importance for the primary horizon (30d)
# and threshold (fixed_1%) to keep runtime reasonable.

def compute_wfv_feature_importance(X, y, model_instance, param_dist, feature_cols,
                                   initial_train_size=0.6, step=30, n_iter=2, cv=3):
    """Average gain-based XGBoost feature importances across walk-forward folds."""
    X_np = _to_np(X)
    y_np = np.asarray(y)
    fold_importances = []
    n = len(X_np)
    idx = int(initial_train_size * n)

    while idx < n:
        X_train_np, X_test_np = X_np[:idx], X_np[idx:idx + step]
        y_train_np = y_np[:idx]
        if len(X_test_np) == 0:
            break

        y_train_encoded = encode_labels(y_train_np)

        inner_cv = TimeSeriesSplit(n_splits=cv)
        search = RandomizedSearchCV(model_instance, param_dist, n_iter=n_iter, cv=inner_cv, scoring='f1_macro', random_state=42)
        try:
            search.fit(X_train_np, y_train_encoded)
            model = search.best_estimator_
            fold_importances.append(model.feature_importances_)
        except Exception:
            pass

        idx += step

    if not fold_importances:
        return None

    avg_importances = np.mean(fold_importances, axis=0)
    X_full = pd.DataFrame(X_np, columns=feature_cols)
    corr_with_signal = [X_full[col].corr(pd.Series(y_np).astype(float)) for col in feature_cols]

    return pd.DataFrame({
        "feature": feature_cols,
        "wfv_xgb_importance": avg_importances,
        "signal_correlation": corr_with_signal,
    }).sort_values("wfv_xgb_importance", ascending=False)


primary_horizon = 30
primary_threshold = "fixed_1%"

for crypto, df, _feature_cols in [("BTC", btc_df, btc_feature_cols), ("ADA", ada_df, ada_feature_cols)]:
    print("\n" + "=" * 100)
    print(f"FEATURE IMPORTANCE (Walk-Forward Validation) - {crypto}")
    print(f"Primary Configuration: {primary_horizon}d horizon, {primary_threshold} threshold")
    print("=" * 100)

    h = primary_horizon
    name = primary_threshold

    signal_col = f"signal_{h}d_{name}"
    feature_cols_list = get_feature_columns(df)
    X = df[feature_cols_list]
    y = df[signal_col]
    combined = pd.concat([X, y], axis=1).dropna()
    X_clean = combined[feature_cols_list]
    y_clean = combined[signal_col].astype(int)

    if len(X_clean) < 100:
        print(f"Insufficient data ({len(X_clean)} samples)")
        continue

    xgb_model = XGBClassifier(
        objective="multi:softprob",
        eval_metric="mlogloss",
        random_state=42,
        tree_method="hist",
    )
    xgb_param_dist = {
        "n_estimators": [200, 400],
        "learning_rate": [0.05, 0.1],
        "max_depth": [3, 5, 7],
        "subsample": [0.7, 1.0],
        "colsample_bytree": [0.7, 1.0],
    }

    feat_imp = compute_wfv_feature_importance(X_clean, y_clean, xgb_model, xgb_param_dist, feature_cols_list)

    if feat_imp is not None:
        print(f"\nTop 15 Most Important Features for {crypto} {h}d {name}:")
        print(feat_imp[["feature", "wfv_xgb_importance", "signal_correlation"]].head(15).to_string(index=False))
    else:
        print("Could not compute feature importance (insufficient folds)")


FEATURE IMPORTANCE (Walk-Forward Validation) - BTC
Primary Configuration: 30d horizon, fixed_1% threshold



Top 15 Most Important Features for BTC 30d fixed_1%:
             feature  wfv_xgb_importance  signal_correlation
     HashRate_30d_MA            0.058969            0.022206
 Price_Dist_lag_182d            0.057801           -0.056225
          CapMVRVCur            0.051408           -0.064393
 NVT_Tx_Basis_lag_1d            0.046559           -0.188916
  Price_Dist_lag_90d            0.044371            0.071345
NVT_Tx_Basis_lag_14d            0.041187           -0.211831
NVT_Tx_Basis_lag_30d            0.038937           -0.212210
        NVT_Tx_Basis            0.037496           -0.188796
 NVT_Tx_Basis_lag_7d            0.035530           -0.202039
      Volatility_30d            0.033454            0.054172
      Puell_Multiple            0.031421           -0.015617
      FearGreedValue            0.022958            0.017589
 Tx_Intensity_lag_7d            0.022524            0.072627
    AdrActCnt_lag_3d            0.022509           -0.013774
  Price_Dist_lag_30d           


Top 15 Most Important Features for ADA 30d fixed_1%:
             feature  wfv_xgb_importance  signal_correlation
             SplyCur            0.094685           -0.125494
      ActiveStakeADA            0.060523           -0.087021
        StakingRatio            0.053980           -0.032551
NVT_Tx_Basis_lag_30d            0.052142            0.023617
 Price_Dist_lag_182d            0.040906            0.092579
 NVT_Tx_Basis_lag_7d            0.038556            0.074642
 NVT_Tx_Basis_lag_1d            0.038247            0.089329
      Volatility_30d            0.036759           -0.086848
NVT_Tx_Basis_lag_14d            0.035814            0.054225
  Price_Dist_lag_90d            0.035425            0.162919
        NVT_Tx_Basis            0.034889            0.090234
          CapMVRVCur            0.034444            0.007532
  Price_Dist_lag_30d            0.031147            0.041867
    AdrActCnt_lag_7d            0.031093           -0.203756
 NVT_Tx_Basis_lag_3d           

In [11]:
%%time
# ============================================================================
# BINARY MCC EVALUATION - USING WALK-FORWARD VALIDATION PREDICTIONS
# ============================================================================
print("Evaluating Binary MCC using Walk-Forward Validation predictions...")
print("Configuration: BTC, 30-day horizon, 1% fixed threshold\n")

sample_horizon = 30
sample_threshold = 0.01

btc_df[f"signal_{sample_horizon}d_sample"] = btc_df[f"fwd_return_{sample_horizon}d"].apply(
    lambda x: classify_signal(x, sample_threshold)
)

feature_cols_mcc = get_feature_columns(btc_df)

X_btc = btc_df[feature_cols_mcc]
y_btc = btc_df[f"signal_{sample_horizon}d_sample"]

combined = pd.concat([X_btc, y_btc], axis=1).dropna()
X_clean = combined[feature_cols_mcc]
y_clean = combined[f"signal_{sample_horizon}d_sample"].astype(int)

print(f"Dataset size: {len(X_clean)} samples")
print("Signal distribution (Training):")
print(f"  - Buy (1):  {(y_clean == 1).sum()} samples ({(y_clean == 1).sum()/len(y_clean)*100:.1f}%)")
print(f"  - Hold (0): {(y_clean == 0).sum()} samples ({(y_clean == 0).sum()/len(y_clean)*100:.1f}%)")
print(f"  - Sell (-1): {(y_clean == -1).sum()} samples ({(y_clean == -1).sum()/len(y_clean)*100:.1f}%)")

print("\nRunning walk-forward validation to collect predictions...")
xgb_model = XGBClassifier(
    objective="multi:softprob",
    eval_metric="mlogloss",
    random_state=42,
    tree_method="hist",
)
xgb_param_dist = {
    "n_estimators": [200, 400],
    "learning_rate": [0.05, 0.1],
    "max_depth": [3, 5, 7],
    "subsample": [0.7, 1.0],
    "colsample_bytree": [0.7, 1.0],
}

avg_metrics_wfv, y_true_wfv, y_pred_wfv = walk_forward_with_predictions(
    X_clean, y_clean, xgb_model, xgb_param_dist,
    initial_train_size=0.6, step=30, n_iter=2, cv=3,
)

print("Walk-forward validation completed!")
print(f"Total predictions collected: {len(y_pred_wfv)} across all folds")

if len(y_pred_wfv) == 0:
    print("No predictions collected (insufficient test windows). Try reducing initial_train_size or the step size.")
else:
    print(f"\nWalk-Forward Prediction distribution:")
    print(f"  - Buy (1):  {sum(p == 1 for p in y_pred_wfv)} samples ({(sum(p == 1 for p in y_pred_wfv)/len(y_pred_wfv))*100:.1f}%)")
    print(f"  - Hold (0): {sum(p == 0 for p in y_pred_wfv)} samples ({(sum(p == 0 for p in y_pred_wfv)/len(y_pred_wfv))*100:.1f}%)")
    print(f"  - Sell (-1): {sum(p == -1 for p in y_pred_wfv)} samples ({(sum(p == -1 for p in y_pred_wfv)/len(y_pred_wfv))*100:.1f}%)")
    print("\n" + "=" * 80)
    print("BINARY MCC EVALUATION (Walk-Forward Validation)")
    print("=" * 80)
    mcc_results = evaluate_signal_quality(y_true_wfv, y_pred_wfv, verbose=True)


Evaluating Binary MCC using Walk-Forward Validation predictions...
Configuration: BTC, 30-day horizon, 1% fixed threshold

Dataset size: 1949 samples
Signal distribution (Training):
  - Buy (1):  963 samples (49.4%)
  - Hold (0): 117 samples (6.0%)
  - Sell (-1): 869 samples (44.6%)

Running walk-forward validation to collect predictions...


Walk-forward validation completed!
Total predictions collected: 780 across all folds

Walk-Forward Prediction distribution:
  - Buy (1):  450 samples (57.7%)
  - Hold (0): 6 samples (0.8%)
  - Sell (-1): 324 samples (41.5%)

BINARY MCC EVALUATION (Walk-Forward Validation)

Binary MCC for Buy Signal (Class 1)
MCC: 0.0914
Weighted MCC components (Buy):
  [OK] TP: 238, [OK] TN: 186
  [X] Opposite FP (pred Buy | true Sell): 179 (x2.0)
  [X] Opposite FN (pred Sell | true Buy): 140 (x2.0)
  [X] Miss from Hold (true Buy | pred Hold): 4
  [X] False Buy from Hold (true Hold | pred Buy): 33
Weighted MCC (Buy): -0.2238

Binary MCC for Sell Signal (Class -1)
MCC: 0.0993
Weighted MCC components (Sell):
  [OK] TP: 161, [OK] TN: 275
  [X] Opposite FP (pred Sell | true Buy): 140 (x2.0)
  [X] Opposite FN (pred Buy | true Sell): 179 (x2.0)
  [X] Miss from Hold (true Sell | pred Hold): 2
  [X] False Sell from Hold (true Hold | pred Sell): 23
Weighted MCC (Sell): -0.2176
CPU times: total: 19min 48s
Wall t

In [12]:
# ============================================================================
# PHASE 0: PER-FOLD STABILITY (never trust a single pooled number)
# ============================================================================
# The pooled MCC above uses every walk-forward fold at once. Here we recompute
# the metrics fold-by-fold (chunking the collected predictions) and report
# mean +/- std. A std comparable to the mean means the signal is not stable.
from sklearn.metrics import f1_score as _f1, matthews_corrcoef as _mcc

if len(y_pred_wfv) > 0:
    _step = 30
    _yt = np.array(y_true_wfv)
    _yp = np.array(y_pred_wfv)
    _folds = [(_yt[i:i + _step], _yp[i:i + _step]) for i in range(0, len(_yt), _step)]
    _f1b, _f1s, _mb, _ms = [], [], [], []
    for _t, _p in _folds:
        _f1b.append(_f1(_t, _p, labels=[1], average='macro', zero_division=0))
        _f1s.append(_f1(_t, _p, labels=[-1], average='macro', zero_division=0))
        for _lab, _store in ((1, _mb), (-1, _ms)):
            _tb = (_t == _lab).astype(int)
            _pb = (_p == _lab).astype(int)
            if len(np.unique(_tb)) > 1 and len(np.unique(_pb)) > 1:
                _store.append(_mcc(_tb, _pb))
    print("\n" + "=" * 80)
    print("PER-FOLD STABILITY  (BTC 30d/1%, mean +/- std across walk-forward folds)")
    print("=" * 80)
    print(f"Folds: {len(_folds)} (step={_step})")
    print(f"Buy  F1 : {np.mean(_f1b):.4f} +/- {np.std(_f1b):.4f}")
    print(f"Sell F1 : {np.mean(_f1s):.4f} +/- {np.std(_f1s):.4f}")
    print(f"Buy  MCC: {np.mean(_mb):.4f} +/- {np.std(_mb):.4f}  (defined in {len(_mb)}/{len(_folds)} folds)")
    print(f"Sell MCC: {np.mean(_ms):.4f} +/- {np.std(_ms):.4f}  (defined in {len(_ms)}/{len(_folds)} folds)")
    print("NOTE: std comparable to the mean => signal not stable fold-to-fold;")
    print("      MCC undefined in folds where Buy or Sell is absent from the 30-day window.")



PER-FOLD STABILITY  (BTC 30d/1%, mean +/- std across walk-forward folds)
Folds: 26 (step=30)
Buy  F1 : 0.4669 +/- 0.3020
Sell F1 : 0.3425 +/- 0.3158
Buy  MCC: 0.2422 +/- 0.2023  (defined in 15/26 folds)
Sell MCC: 0.2465 +/- 0.1966  (defined in 15/26 folds)
NOTE: std comparable to the mean => signal not stable fold-to-fold;
      MCC undefined in folds where Buy or Sell is absent from the 30-day window.


In [13]:
# ============================================================================
# MODEL COMPARISON: XGBOOST vs LOGISTIC REGRESSION
# ============================================================================
# LR baseline below is the fresh leak-free run of 2a_lr_model_pipeline.ipynb
# (refreshed 2026-06-22), averaged across all 24 configurations. Dataset spans
# 2020-01-01..2026-06-01 (2344 rows/asset; each config uses fewer after dropna).

print("\n" + "=" * 80)
print("MODEL COMPARISON: XGBOOST vs LOGISTIC REGRESSION")
print("=" * 80)

xgb_precision_buy  = all_results_df['avg_precision_buy'].mean()
xgb_recall_buy     = all_results_df['avg_recall_buy'].mean()
xgb_f1_buy         = all_results_df['avg_f1_buy'].mean()
xgb_precision_sell = all_results_df['avg_precision_sell'].mean()
xgb_recall_sell    = all_results_df['avg_recall_sell'].mean()
xgb_f1_sell        = all_results_df['avg_f1_sell'].mean()

# LR baseline from 2a notebook (fresh leak-free run, 24-config averages)
lr_baseline = {
    'precision_buy':  0.2936, 'recall_buy':  0.2458, 'f1_buy':  0.2144,
    'precision_sell': 0.3625, 'recall_sell': 0.2588, 'f1_sell': 0.2455,
}

print("\nAverage performance across 24 configurations")
print(f"  XGBOOST  Buy  - Prec: {xgb_precision_buy:.4f}, Rec: {xgb_recall_buy:.4f}, F1: {xgb_f1_buy:.4f}")
print(f"  XGBOOST  Sell - Prec: {xgb_precision_sell:.4f}, Rec: {xgb_recall_sell:.4f}, F1: {xgb_f1_sell:.4f}")
print(f"  LR       Buy  - Prec: {lr_baseline['precision_buy']:.4f}, Rec: {lr_baseline['recall_buy']:.4f}, F1: {lr_baseline['f1_buy']:.4f}")
print(f"  LR       Sell - Prec: {lr_baseline['precision_sell']:.4f}, Rec: {lr_baseline['recall_sell']:.4f}, F1: {lr_baseline['f1_sell']:.4f}")

buy_delta  = xgb_f1_buy  - lr_baseline['f1_buy']
sell_delta = xgb_f1_sell - lr_baseline['f1_sell']
buy_pct  = (buy_delta  / lr_baseline['f1_buy']  * 100) if lr_baseline['f1_buy']  > 0 else 0
sell_pct = (sell_delta / lr_baseline['f1_sell'] * 100) if lr_baseline['f1_sell'] > 0 else 0
print("\nXGBoost - LR delta (F1):")
print(f"  Buy:  {xgb_f1_buy:.4f} vs {lr_baseline['f1_buy']:.4f}  -> +{buy_delta:.4f} ({buy_pct:+.1f}%)")
print(f"  Sell: {xgb_f1_sell:.4f} vs {lr_baseline['f1_sell']:.4f} -> +{sell_delta:.4f} ({sell_pct:+.1f}%)")

print("\nXGBoost F1 by horizon:")
for h in [3, 7, 14, 30]:
    horizon_data = all_results_df[all_results_df['horizon_days'] == h]
    print(
        f"  {h:>2}d: Buy F1={horizon_data['avg_f1_buy'].mean():.4f}, "
        f"Sell F1={horizon_data['avg_f1_sell'].mean():.4f}"
    )

print("\nTop 3 XGBoost configurations by Buy F1:")
top_configs = all_results_df.nlargest(3, 'avg_f1_buy')[
    ['crypto', 'horizon_days', 'threshold_type', 'avg_f1_buy', 'avg_precision_buy', 'avg_recall_buy']
]
for _, row in top_configs.iterrows():
    print(
        f"  {row['crypto']} {row['horizon_days']:>2}d {row['threshold_type']:12s}: "
        f"F1={row['avg_f1_buy']:.4f} (Prec={row['avg_precision_buy']:.4f}, Rec={row['avg_recall_buy']:.4f})"
    )

print("=" * 80)



MODEL COMPARISON: XGBOOST vs LOGISTIC REGRESSION

Average performance across 24 configurations
  XGBOOST  Buy  - Prec: 0.4224, Rec: 0.4864, F1: 0.3734
  XGBOOST  Sell - Prec: 0.4999, Rec: 0.5319, F1: 0.4353
  LR       Buy  - Prec: 0.2936, Rec: 0.2458, F1: 0.2144
  LR       Sell - Prec: 0.3625, Rec: 0.2588, F1: 0.2455

XGBoost - LR delta (F1):
  Buy:  0.3734 vs 0.2144  -> +0.1590 (+74.2%)
  Sell: 0.4353 vs 0.2455 -> +0.1898 (+77.3%)

XGBoost F1 by horizon:
   3d: Buy F1=0.3523, Sell F1=0.3883
   7d: Buy F1=0.3893, Sell F1=0.4522
  14d: Buy F1=0.3852, Sell F1=0.4666
  30d: Buy F1=0.3670, Sell F1=0.4341

Top 3 XGBoost configurations by Buy F1:
  BTC 14d fixed_0.5%  : F1=0.4960 (Prec=0.5277, Rec=0.6580)
  BTC 30d fixed_0.5%  : F1=0.4764 (Prec=0.4686, Rec=0.6239)
  BTC  7d fixed_0.5%  : F1=0.4713 (Prec=0.4869, Rec=0.6470)


## Final Analysis: XGBoost vs Logistic Regression

> **Numbers refreshed 2026-06-22** from a clean end-to-end re-execution (leak-free per-fold protocol — XGBoost is scale-invariant so scaling is skipped — no synthetic Hold labels, `TimeSeriesSplit` inner CV). Data: 2020-01-01 → 2026-06-01 (2344 rows/asset; 1949 usable for BTC 30d/1% after `dropna`). Seeds pinned (`random_state=42`).

### Conventional metrics (24-config averages ± spread across configs)

| Metric | Logistic Regression | XGBoost | Delta |
|--------|---------------------|---------|-------|
| Buy F1  | 0.214 ± 0.081 | 0.373 ± 0.071 | +74% |
| Sell F1 | 0.246 ± 0.104 | 0.435 ± 0.126 | +77% |

XGBoost clearly beats the linear baseline as a *classifier* — the non-linearity hypothesis holds.

### Binary MCC — BTC 30d / 1% threshold

| Metric | Logistic Regression | XGBoost |
|--------|---------------------|---------|
| Buy MCC           | 0.0476  | 0.0914  |
| Buy Weighted MCC  | −0.1726 | −0.2238 |
| Sell MCC          | 0.0574  | 0.0993  |
| Sell Weighted MCC | −0.1541 | −0.2176 |

### Never one number — per-fold stability (XGBoost, BTC 30d/1%, 26 folds)

| Metric | Mean ± std | Folds defined |
|--------|-----------|---------------|
| Buy F1  | 0.467 ± 0.302 | 26/26 |
| Sell F1 | 0.343 ± 0.316 | 26/26 |
| Buy MCC  | 0.242 ± 0.202 | 15/26 |
| Sell MCC | 0.247 ± 0.197 | 15/26 |

XGBoost's per-fold MCC (~0.24) is genuinely higher than LR's (~0.01–0.09), but **the std still ≈ the mean** and MCC is undefined in ~40% of folds (Buy or Sell absent from the 30-day window). Even the stronger model is unstable fold-to-fold.

### The key nuance: better classifier, worse weighted MCC

- LR predicted **Hold 40.6%** of the time — it hedged.
- XGBoost predicted **Hold 0.8%** — it almost always commits to Buy or Sell.

Because opposite-direction errors are penalised 2×, XGBoost's decisiveness backfires (319 wrong-direction calls: 179 pred-Buy/true-Sell + 140 pred-Sell/true-Buy). So XGBoost is the stronger classifier but **more confidently wrong in the costliest way**. Both models have **negative weighted MCC** — neither is a deployable directional signal.

### Patterns

- **Sell beats Buy** on average (Sell F1 0.435 vs Buy 0.373).
- **Best horizons: 7d–14d** (Sell F1 0.452 / 0.467); 3d noisier, 30d decays.
- **Best single config: ADA 7d / 0.5% → Sell F1 0.620** — but this is the best of 24; treat as a multiple-testing candidate, not a confirmed edge (`VALIDATION_PLAN.md` Phase 2.3).

### Feature importance

- **BTC**: `HashRate_30d_MA`, `Price_Dist_lag_182d`, `CapMVRVCur` lead; NVT lags carry the strongest signal correlation (~−0.21).
- **ADA**: top 3 are `SplyCur`, `ActiveStakeADA`, `StakingRatio` — the Blockfrost staking/supply data dominates ADA prediction. ⚠️ Audit the epoch→daily staking expansion for look-ahead before trusting ADA's edge (`VALIDATION_PLAN.md` Phase 3.3).

### Bottom line (Phase 0)

Numbers are now trustworthy and reproducible. On this evidence, **neither model is a tradeable directional signal** (near-zero, unstable MCC; negative cost-weighted MCC). XGBoost is the better classifier and the right base for further work. The follow-up validation (ablation, model-free signal detection, and the volatility / cycle / regime studies) has since been completed and confirms this: no on-chain directional edge over price — see `docs/CONCLUSIONS.md`.
